In [1]:
# import pandas as pd
# housing = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

import polars as pl
housing = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

In [4]:
housing.shape
housing.head()

id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
i64,str,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64
1565930130,"""20141104T000000""",4,3.25,3760,4675,2.0,0,0,3,8,2740,1020,2007,0,98038,47.3862,-122.048,3280,4033,429900.0
3279000420,"""20150115T000000""",3,1.75,1460,7800,1.0,0,0,2,7,1040,420,1979,0,98023,47.3035,-122.382,1310,7865,233000.0
194000575,"""20141014T000000""",4,1.0,1340,5800,1.5,0,2,3,7,1340,0,1914,0,98116,47.5658,-122.389,1900,5800,455000.0
2115510160,"""20141208T000000""",3,1.75,1440,8050,1.0,0,0,3,8,1440,0,1985,0,98023,47.3187,-122.39,1790,7488,258950.0
7522500005,"""20140815T000000""",2,1.5,1780,4750,1.0,0,0,4,7,1080,700,1947,0,98117,47.6859,-122.395,1690,5962,555000.0


In [19]:
df = housing.select('zipcode', 'grade', 'condition', 'view', 'floors', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'sqft_living15', 'sqft_lot15', 'long', 'lat', 'price')

# , (pl.col('sqft_living')-pl.col('sqft_living15')).alias('living_diff'), (pl.col('sqft_lot')-pl.col('sqft_lot15')).alias('lot_diff')

df.head()

zipcode,grade,condition,view,floors,bedrooms,bathrooms,sqft_living,sqft_lot,sqft_living15,sqft_lot15,long,lat,price
i64,i64,i64,i64,f64,i64,f64,i64,i64,i64,i64,f64,f64,f64
98038,8,3,0,2.0,4,3.25,3760,4675,3280,4033,-122.048,47.3862,429900.0
98023,7,2,0,1.0,3,1.75,1460,7800,1310,7865,-122.382,47.3035,233000.0
98116,7,3,2,1.5,4,1.0,1340,5800,1900,5800,-122.389,47.5658,455000.0
98023,8,3,0,1.0,3,1.75,1440,8050,1790,7488,-122.39,47.3187,258950.0
98117,7,4,0,1.0,2,1.5,1780,4750,1690,5962,-122.395,47.6859,555000.0


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X = df.to_numpy()
y = housing['price'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeRegressor(max_depth=8, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, preds))
print('R²: ', r2_score(y_test, preds))

MAE: 132199.04987239206
R²:  0.6864214136906497


In [16]:
#Importance Table
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(1000)

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(pl.DataFrame({
        'feature': df.columns,
        'importance': model.feature_importances_,
    }).sort('importance', descending=True))

shape: (11, 2)
┌───────────────┬────────────┐
│ feature       ┆ importance │
│ ---           ┆ ---        │
│ str           ┆ f64        │
╞═══════════════╪════════════╡
│ grade         ┆ 0.451599   │
│ sqft_living   ┆ 0.293516   │
│ zipcode       ┆ 0.115762   │
│ view          ┆ 0.045592   │
│ sqft_living15 ┆ 0.036161   │
│ sqft_lot15    ┆ 0.020906   │
│ bathrooms     ┆ 0.02039    │
│ sqft_lot      ┆ 0.011973   │
│ condition     ┆ 0.003089   │
│ bedrooms      ┆ 0.000761   │
│ floors        ┆ 0.000251   │
└───────────────┴────────────┘
